In [1]:
%pip install numpy
%pip install pandas
%pip install scikit-learn
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1.2 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.1.2 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: scikit-learn in c:\python312\lib\site-packages (1.6.1)




[notice] A new release of pip is available: 24.1.2 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.1.2 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
# from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
df= pd.read_csv("Cleaned_Indian_Food_Dataset.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5938 entries, 0 to 5937
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   TranslatedRecipeName    5938 non-null   object
 1   TranslatedIngredients   5938 non-null   object
 2   TotalTimeInMins         5938 non-null   int64 
 3   Cuisine                 5938 non-null   object
 4   TranslatedInstructions  5938 non-null   object
 5   URL                     5938 non-null   object
 6   Cleaned-Ingredients     5938 non-null   object
 7   image-url               5938 non-null   object
 8   Ingredient-count        5938 non-null   int64 
dtypes: int64(2), object(7)
memory usage: 417.6+ KB


In [5]:
df.isnull().sum()

TranslatedRecipeName      0
TranslatedIngredients     0
TotalTimeInMins           0
Cuisine                   0
TranslatedInstructions    0
URL                       0
Cleaned-Ingredients       0
image-url                 0
Ingredient-count          0
dtype: int64

In [6]:
train = df.drop(['TranslatedIngredients', 'TotalTimeInMins', 'Cuisine', 'URL', 'image-url'], axis=1)

In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5938 entries, 0 to 5937
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   TranslatedRecipeName    5938 non-null   object
 1   TranslatedInstructions  5938 non-null   object
 2   Cleaned-Ingredients     5938 non-null   object
 3   Ingredient-count        5938 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 185.7+ KB


In [8]:
train.head()

,TranslatedRecipeName,TranslatedInstructions,Cleaned-Ingredients,Ingredient-count
0,Masala Karela Recipe,"To begin making the Masala Karela Recipe,de-se...","salt,amchur (dry mango powder),karela (bitter ...",10
1,Spicy Tomato Rice (Recipe),"To make tomato puliogere, first cut the tomato...","tomato,salt,chickpea lentils,green chilli,rice...",12
2,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,"To begin making the Ragi Vermicelli Recipe, fi...","salt,rice vermicelli noodles (thin),asafoetida...",12
3,Gongura Chicken Curry Recipe - Andhra Style Go...,To begin making Gongura Chicken Curry Recipe f...,"tomato,salt,ginger,sorrel leaves (gongura),fen...",15
4,Andhra Style Alam Pachadi Recipe - Adrak Chutn...,"To make Andhra Style Alam Pachadi, first heat ...","tomato,salt,ginger,red chillies,curry,asafoeti...",12


In [9]:
# 1. Check for devices (CPU, GPU)
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    device = "/GPU:0"
    print("GPU found, using GPU for training")
else:
    device = "/CPU:0"
    print("No GPU found, using CPU for training")


No GPU found, using CPU for training


In [10]:
# Tokenize the text
max_vocab_size = 10000  # Choose an appropriate size
tokenizer = Tokenizer(num_words=max_vocab_size)
tokenizer.fit_on_texts(train['TranslatedInstructions'])
total_words = len(tokenizer.word_index) + 1

# Create sequences (same as before)
input_sequences = []
for line in train['TranslatedInstructions']:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences (same as before)
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

# Prepare input (X) - the output (y) will be the token IDs directly
X = input_sequences[:, :-1]
y = input_sequences[:, -1] # Target is the next token ID (not one-hot encoded)

# Use sparse categorical cross-entropy to avoid one-hot encoding
print(f"Total vocabulary size: {total_words}")
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Total vocabulary size: 10721
Shape of X: (1268464, 1034)
Shape of y: (1268464,)


In [11]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

def create_model(total_words, max_sequence_len):
    model = Sequential()
    model.add(Embedding(input_dim=total_words, output_dim=100, input_length=max_sequence_len - 1))  # Specify input_length
    model.add(LSTM(128, return_sequences=True))
    model.add(LSTM(128))
    model.add(Dense(total_words, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  # Use sparse_categorical_crossentropy
    return model

# Create the model

In [12]:
model = create_model(total_words, max_sequence_len)

# Explicitly build the model
model.build(input_shape=(None, max_sequence_len - 1))

# Print model summary
print("Model Summary BEFORE Training:")
model.summary()

C:\Users\KIIT\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model Summary BEFORE Training:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 1034, 100)      │     1,072,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 1034, 128)      │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10721)          │     1,383,009 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,703,941 (10.31 MB)

 Trainable params: 2,703,941 (10.31 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import time

start_time = time.time()
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model.fit(X_train, y_train, epochs=25, batch_size=1024, verbose=1, validation_data=(X_val, y_val), callbacks=[early_stopping])
end_time = time.time()

print(f"Training time: {(end_time - start_time) / 60:.2f} minutes")

Epoch 1/25
991/991 ━━━━━━━━━━━━━━━━━━━━ 21151s 21s/step - accuracy: 0.0749 - loss: 6.3391 - val_accuracy: 0.1162 - val_loss: 5.6718
Epoch 2/25
991/991 ━━━━━━━━━━━━━━━━━━━━ 26079s 26s/step - accuracy: 0.1430 - loss: 5.3520 - val_accuracy: 0.2107 - val_loss: 4.6689
Epoch 3/25
991/991 ━━━━━━━━━━━━━━━━━━━━ 29239s 29s/step - accuracy: 0.2252 - loss: 4.5076 - val_accuracy: 0.2602 - val_loss: 4.2207
Epoch 4/25
991/991 ━━━━━━━━━━━━━━━━━━━━ 17404s 18s/step - accuracy: 0.2705 - loss: 4.1039 - val_accuracy: 0.2888 - val_loss: 3.9722
Epoch 5/25
991/991 ━━━━━━━━━━━━━━━━━━━━ 19034s 19s/step - accuracy: 0.2989 - loss: 3.8624 - val_accuracy: 0.3099 - val_loss: 3.8130
Epoch 6/25
795/991 ━━━━━━━━━━━━━━━━━━━━ 51:13 16s/step - accuracy: 0.3181 - loss: 3.7005

In [ ]:
# # Split the data into training and validation sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Train the model
# model.fit(X_train, y_train, epochs=10, batch_size=256, validation_data=(X_test, y_test), verbose=1)
# # Evaluate the model        
# loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
# print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
# %pip install nltk
# %pip install gensim

In [ ]:
# x = train['Cleaned-Ingredients']  
# y = train['TranslatedInstructions'] 

# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [ ]:
# from gensim.models import Word2Vec
# from nltk.tokenize import word_tokenize
# import nltk
# nltk.download('punkt')  # Download the tokenizer models if not already downloaded

# #tokenized_corpus = word_tokenize(x_train.lower())  # Lowercasing for consistency
 
# skipgram_model = Word2Vec(sentences=[x_train],
#                           vector_size=100,  # Dimensionality of the word vectors
#                           window=5,         # Maximum distance between the current and predicted word within a sentence
#                           sg=1,             # Skip-Gram model (1 for Skip-Gram, 0 for CBOW)
#                           min_count=1,      # Ignores all words with a total frequency lower than this
#                           workers=4)        # Number of CPU cores to use for training the model

# skipgram_model.train([x_train], total_examples=1, epochs=10)
# skipgram_model.save("skipgram_model.model")
# loaded_model = Word2Vec.load("skipgram_model.model")
# vector_representation = loaded_model.wv['word']